In [3]:
import simnibs
import simnibs.segmentation
from simnibs.segmentation import charm_main
import numpy as np
import pandas as pd
import os

In [3]:
def create_head_model(sub_id , output_dir, t1_path, t2_path=None):
    charm_main.run(
    subject_dir = output_dir,
    T1 = t1_path,
    T2 = t2_path,
    initatlas = True,       # Improve atlas alignment
    segment = True,         # Run segmentation
    create_surfaces = True, # Generate surfaces
    mesh_image = True,     # Skip meshing (do later with simnibs.mesh_tools)
    
    )

In [5]:
def create_varied_conductivity_models(base_mesh, output_dir, n_models=5):
    """Create multiple models with varied conductivity values"""
    conductivity_ranges = {
        'WM': (0.08, 0.16), 'GM': (0.20, 0.35), 'CSF': (1.5, 1.8),
        'Bone': (0.008, 0.012), 'Scalp': (0.4, 0.5)
    }
    
    models = []
    for i in range(n_models):
        mesh = base_mesh.copy()
        conductivity = {tissue: np.random.uniform(low, high) 
                       for tissue, (low, high) in conductivity_ranges.items()}
        
        for tissue, cond in conductivity.items():
            mesh.add_element_field(cond, f"conductivity_{tissue}")
        
        model_path = os.path.join(output_dir, f"model_variation_{i+1}.msh")
        simnibs.msh.mesh_io.write_msh(mesh, model_path)
        models.append((model_path, conductivity))
    
    return models

In [6]:
def create_3x3_array(center, normal, spacing=15):
    """Create a 3x3 grid of electrodes on a plane"""
    if abs(normal[0]) > 0.5:
        temp = np.array([0, 1, 0])
    else:
        temp = np.array([1, 0, 0])
    
    tangent1 = np.cross(normal, temp)
    tangent1 /= np.linalg.norm(tangent1)
    tangent2 = np.cross(normal, tangent1)
    
    positions = []
    for i in [-1, 0, 1]:
        for j in [-1, 0, 1]:
            offset = (i * spacing * tangent1) + (j * spacing * tangent2)
            positions.append(center + offset)
    
    return np.array(positions)

In [7]:
def get_random_scalp_locations(mesh, min_distance=60, max_attempts=100):
    """Get two random scalp locations that aren't too close"""
    scalp_surface = mesh.crop_mesh(elm_type=2, tag=1005)
    nodes = scalp_surface.nodes.node_coord
    normals = scalp_surface.nodes_normals()
    
    # Pick first random location
    idx1 = np.random.choice(len(nodes))
    center1 = nodes[idx1]
    normal1 = -normals[idx1]  # Flip normal inward
    normal1 /= np.linalg.norm(normal1)
    
    # Pick second location that's sufficiently far away
    attempts = 0
    while attempts < max_attempts:
        idx2 = np.random.choice(len(nodes))
        center2 = nodes[idx2]
        
        # Check distance between array centers
        if np.linalg.norm(center1 - center2) >= min_distance:
            normal2 = -normals[idx2]
            normal2 /= np.linalg.norm(normal2)
            return [(center1, normal1), (center2, normal2)]
        
        attempts += 1
    
    # If we can't find a good pair, just return any two locations
    idx2 = (idx1 + len(nodes)//2) % len(nodes)  # At least pick something opposite-ish
    center2 = nodes[idx2]
    normal2 = -normals[idx2]
    normal2 /= np.linalg.norm(normal2)
    return [(center1, normal1), (center2, normal2)]

In [8]:
def add_electrode_arrays(mesh, array_locations):
    """Add electrode arrays to the mesh"""
    all_electrodes = []
    
    for array_idx, (center, normal) in enumerate(array_locations):
        positions = create_3x3_array(center, normal)
        
        for i, pos in enumerate(positions):
            el = simnibs.electrode_placements.Electrode()
            el.centre = pos
            el.shape = 'ellipse'
            el.dimensions = [10, 10]  # 10mm diameter
            el.thickness = [1]  # 1mm thickness
            el.channelnr = array_idx * 9 + i + 1  # Unique channel numbers
            all_electrodes.append(el)
    
    mesh_with_elec = simnibs.electrode_placements.add_electrodes_to_mesh(
        mesh, all_electrodes, add_to_skin=True
    )
    
    return mesh_with_elec, all_electrodes

In [9]:
def process_Subject(subject_ID , t1_path , t2_path):
    # Setup paths
    m2m_dir = "E:/Datasets/Stage 2/Segmentation"
    m2m_dir = os.path.join(m2m_dir, f"{subject_ID}")
    os.makedirs(m2m_dir, exist_ok=True)
    output_dir = "E:/Datasets/Stage 2/Meshes"
    output_dir = os.path.join(output_dir, f"{subject_ID}")
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Create base head model
    head_mesh = create_head_model(subject_ID ,m2m_dir, t1_path, t2_path)
    
    # 2. Create conductivity variations
    models = create_varied_conductivity_models(head_mesh, output_dir, n_models=5)
    
    # 3. Process each model
    final_models = []
    for model_path, conductivity in models:
        mesh = simnibs.msh.mesh_io.read_msh(model_path)
        
        # Get random array locations (with minimum distance)
        array_locations = get_random_scalp_locations(mesh)
        
        # Add electrode arrays
        mesh_with_elec, electrodes = add_electrode_arrays(mesh, array_locations)
        
        # Save final model
        final_path = os.path.join(output_dir, f"random_arrays_{os.path.basename(model_path)}")
        simnibs.msh.mesh_io.write_msh(mesh_with_elec, final_path)
        
        # Save electrode positions
        elec_positions = np.array([el.centre for el in electrodes])
        df = pd.DataFrame(elec_positions, columns=['x', 'y', 'z'])
        df['array'] = ['Array1']*9 + ['Array2']*9
        df['electrode'] = [f"E{i+1}" for i in range(18)]
        csv_path = final_path.replace('.msh', '_electrodes.csv')
        df.to_csv(csv_path, index=False)
        
        final_models.append((final_path, conductivity, array_locations))
    
    return final_models

In [4]:
t1 = "E:/Datasets/Coregisterd-rigid/TCGA-02-0006/t1/t1_to_mni.nii.gz"
t2 = "E:/Datasets/Coregisterd-rigid/TCGA-02-0006/t2/t2_to_mni.nii.gz"
#final_models = process_Subject("TCGA-02-0006" , t1 , t2)
m2m_dir = "E:/Datasets/Stage 2/Meshes"
m2m_dir = os.path.join(m2m_dir, "m2m_TCGA-02-0006")
os.makedirs(m2m_dir, exist_ok=True)
create_head_model("TCGA-02-0006", m2m_dir , t1 , t2)
print("finished")
    

[ simnibs ]INFO: simnibs version 4.0.0
[ simnibs ]INFO: charm run started: Fri Jun 20 16:10:17 2025
[ simnibs ]INFO: Reading settings from E:\Anaconda\envs\simnibs_env\Lib\site-packages\simnibs\charm.ini
[ simnibs ]INFO: Using 8 threads, instead of all available.
[ simnibs ]INFO: Using charm_atlas_mni as charm atlas.
[ simnibs ]INFO: Reading settings from E:\Anaconda\envs\simnibs_env\Lib\site-packages\simnibs\segmentation\atlases\charm_atlas_mni\charm_atlas_mni.ini
[ simnibs ]INFO: Starting affine registration and neck correction.


performing affine atlas registration
image: E:/Datasets/Coregisterd-rigid/TCGA-02-0006/t1/t1_to_mni.nii.gz
template: E:\Anaconda\envs\simnibs_env\Lib\site-packages\simnibs\segmentation\atlases\charm_atlas_mni\template.nii
maximalDeformation=1.0000 minLogLikelihood=-0.7337
maximalDeformation=1.0332 minLogLikelihood=-0.7417
maximalDeformation=1.1098 minLogLikelihood=-0.7424
maximalDeformation=1.2213 minLogLikelihood=-0.7446
maximalDeformation=0.2871 minLogLikelihood=-0.7450
maximalDeformation=0.1032 minLogLikelihood=-0.7450
maximalDeformation=0.1862 minLogLikelihood=-0.7452
maximalDeformation=0.1169 minLogLikelihood=-0.7452
maximalDeformation=0.1683 minLogLikelihood=-0.7452
maximalDeformation=1.2240 minLogLikelihood=-0.7453
maximalDeformation=0.5971 minLogLikelihood=-0.7456
maximalDeformation=1.4553 minLogLikelihood=-0.7468
maximalDeformation=1.0124 minLogLikelihood=-0.7472
maximalDeformation=0.3213 minLogLikelihood=-0.7482
maximalDeformation=0.3860 minLogLikelihood=-0.7486
maximalDeform

[ simnibs ]INFO: Template registration summary.
[ simnibs ]INFO: Number of Iterations: 5, Cost: -0.820949

[ simnibs ]INFO: Adjusting neck.


['E:/Datasets/Coregisterd-rigid/TCGA-02-0006/t1/t1_to_mni.nii.gz']
E:\Datasets\Stage 2\Meshes\m2m_TCGA-02-0006\segmentation\template_coregistered.mgz


[ simnibs.segmentation.samseg.AffineWholeHead ]INFO: Initial cost: -0.14728120233961012
[ simnibs ]INFO: Neck adjustment done.
[ simnibs ]INFO: Estimating parameters.
[ simnibs ]INFO: Starting segmentation.
[ simnibs.segmentation.samseg.Samseg ]INFO: ##----------------------------------------------
[ simnibs.segmentation.samseg.Samseg ]INFO:               Samsegment Options
[ simnibs.segmentation.samseg.Samseg ]INFO: ##----------------------------------------------
[ simnibs.segmentation.samseg.Samseg ]INFO: output directory:E:\Datasets\Stage 2\Meshes\m2m_TCGA-02-0006\segmentation
[ simnibs.segmentation.samseg.Samseg ]INFO: input images: ['E:/Datasets/Coregisterd-rigid/TCGA-02-0006/t1/t1_to_mni.nii.gz', 'E:\\Datasets\\Stage 2\\Meshes\\m2m_TCGA-02-0006\\T2_reg.nii.gz']
[ simnibs.segmentation.samseg.Samseg ]INFO: transformed template:E:\Datasets\Stage 2\Meshes\m2m_TCGA-02-0006\segmentation\template_coregistered.mgz
[ simnibs.segmentation.samseg.Samseg ]INFO: modelSpecifications:{'FreeSur

...

done
Segmenting, can take a while...
Segmented 1 out of 55 structures.
Segmented 2 out of 55 structures.
Segmented 3 out of 55 structures.
Segmented 4 out of 55 structures.
Segmented 5 out of 55 structures.
Segmented 6 out of 55 structures.
Segmented 7 out of 55 structures.
Segmented 8 out of 55 structures.
Segmented 9 out of 55 structures.
Segmented 10 out of 55 structures.
Segmented 11 out of 55 structures.
Segmented 12 out of 55 structures.
Segmented 13 out of 55 structures.
Segmented 14 out of 55 structures.
Segmented 15 out of 55 structures.
Segmented 16 out of 55 structures.
Segmented 17 out of 55 structures.
Segmented 18 out of 55 structures.
Segmented 19 out of 55 structures.
Segmented 20 out of 55 structures.
Segmented 21 out of 55 structures.
Segmented 22 out of 55 structures.
Segmented 23 out of 55 structures.
Segmented 24 out of 55 structures.
Segmented 25 out of 55 structures.
Segmented 26 out of 55 structures.
Segmented 27 out of 55 structures.
Segmented 28 out of 55 stru

[ simnibs ]INFO: Writing out MNI warps.
[ simnibs ]INFO: Post-processing segmentation
[ simnibs ]INFO: Upsampling bias corrected images.


['E:\\Datasets\\Stage 2\\Meshes\\m2m_TCGA-02-0006\\label_prep\\T1_upsampled.nii.gz', 'E:\\Datasets\\Stage 2\\Meshes\\m2m_TCGA-02-0006\\label_prep\\T2_upsampled.nii.gz']
E:\Datasets\Stage 2\Meshes\m2m_TCGA-02-0006\segmentation\template_coregistered.mgz
done
Segmenting, can take a while...
Segmented 1 out of 55 structures.
Segmented 2 out of 55 structures.
Segmented 3 out of 55 structures.
Segmented 4 out of 55 structures.
Segmented 5 out of 55 structures.
Segmented 6 out of 55 structures.
Segmented 7 out of 55 structures.
Segmented 8 out of 55 structures.
Segmented 9 out of 55 structures.
Segmented 10 out of 55 structures.
Segmented 11 out of 55 structures.
Segmented 12 out of 55 structures.
Segmented 13 out of 55 structures.
Segmented 14 out of 55 structures.
Segmented 15 out of 55 structures.
Segmented 16 out of 55 structures.
Segmented 17 out of 55 structures.
Segmented 18 out of 55 structures.
Segmented 19 out of 55 structures.
Segmented 20 out of 55 structures.
Segmented 21 out of 

[ simnibs ]INFO: Ensure CSF
[ simnibs ]INFO: Starting surface creation
[ simnibs ]INFO: Total time surface creation (HH:MM:SS):
[ simnibs ]INFO: 00:09:33
[ simnibs ]INFO: Improving GM from surfaces
[ simnibs ]INFO: Total time cost for GM imrpovements in (HH:MM:SS):
[ simnibs ]INFO: 00:01:27
[ simnibs ]INFO: Starting mesh
[ simnibs ]INFO: Calculating tissue thickness
[ simnibs ]INFO: Calculating sizing fields
[ simnibs ]INFO: Time to prepare meshing: 00:01:08
[ simnibs ]INFO: Meshing
[ simnibs ]INFO: Time to mesh: 00:08:41
[ simnibs ]INFO: Removing Spikes
[ simnibs ]INFO:  Step 1: Update tags from label image
[ simnibs ]INFO:    Relabled 56659 tets
[ simnibs ]INFO:  Step 2: Update tags from tet neighbors
[ simnibs ]INFO:      It. 0: relabled 13418 tets
[ simnibs ]INFO:      It. 1: relabled 6236 tets
[ simnibs ]INFO:      It. 2: relabled 4212 tets
[ simnibs ]INFO:      It. 3: relabled 3512 tets
[ simnibs ]INFO:      It. 4: relabled 2137 tets
[ simnibs ]INFO:      It. 5: relabled 513 tets

finished


In [ ]:
from simnibs import mni2subject_coords, run_simnibs, read_msh

# 1. Set up the segmentation parameters
sub_id = 'my_subject'  # Subject identifier
t1_file = '/path/to/T1.nii.gz'  # Path to T1-weighted image
t2_file = '/path/to/T2.nii.gz'  # Path to T2-weighted image (optional)

# 2. Run the headreco pipeline (automatic segmentation and mesh creation)
run_simnibs(
    sub_id, 
    t1=t1_file, 
    t2=t2_file,
    options='--cat'  # Use CAT12 for segmentation (recommended)
)

# 3. Load and inspect the resulting mesh
mesh_file = f'{sub_id}/m2m_{sub_id}/{sub_id}.msh'
mesh = read_msh(mesh_file)
print(f"Mesh contains {len(mesh.elm.tags)} elements")
print(f"Tissues segmented: {mesh.tissue_labels}")

In [5]:
help(simnibs.Msh)

Help on class Msh in module simnibs.mesh_tools.mesh_io:

class Msh(builtins.object)
 |  Msh(nodes=None, elements=None, fn=None)
 |  
 |  class to handle the meshes.
 |  Gathers Nodes, Elements and Data
 |  
 |  Parameters
 |  -------------------------
 |  nodes: (optional) simnibs.msh.Nodes
 |      Nodes structure
 |  
 |  elements: (optional) simnibs.msh.Elements()
 |      Elements structure
 |  
 |  fn: str (optional)
 |      Name of ".msh" file to be read.
 |      Overides nodes and elements
 |  
 |  Attributes
 |  -------------------------
 |  nodes: simnibs.msh.Nodes
 |      a Nodes field
 |  elm: simnibs.msh.Elements
 |      A Elements field
 |  nodedata: simnibs.msh.NodeData
 |      list of NodeData filds
 |  elmdata: simnibs.msh.ElementData
 |      list of ElementData fields
 |  fn: str
 |      name of file
 |  binary: bool
 |      wheather or not the mesh was in binary format
 |  
 |  Methods defined here:
 |  
 |  __eq__(self, other)
 |      Return self==value.
 |  
 |  __ini